### Delete Old AlpahCache in Data Dir
### RUN IN CPU or T4 GPU ~01:30:00 to process 1998-2026




In [1]:
# CELL 0: Set Run Parameters
import os

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"

In [2]:
# CELL 1
import time

start_time = time.time()

In [3]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive


In [4]:
# CELL 3: Load Data & Initialize Cache
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from data_pipeline.screener import UniverseScreener
from data_pipeline.cache import CheckpointAlphaCache

print("Loading preprocessed data...")
data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df

# Tripwire validation on processed feature schemas
expected_macro_cols = [
    "Macro_Trend",
    "Macro_Vix_Z",
    "Macro_Vix_Ratio",
    "'Mkt_Vol_63d_Z'",
]
expected_feature_cols = ["Mom_252_21", "RSI", "ATRP"]

assert all(
    col in macro_df.columns for col in expected_macro_cols
), f"macro_df missing expected engineered columns! Found: {macro_df.columns.tolist()}"

assert all(
    col in features_df.columns for col in expected_feature_cols
), f"features_df missing expected columns! Found: {features_df.columns.tolist()}"

print("✅ Data validation passed: Schemas and contracts verified.")

config = TradingConfig()

# Reconstruct wide df_close dynamically and sync calendar
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

NOTEBOOKS_RLVR_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2
PROJECT_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
GLOBAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data
GLOBAL_PROCESSED_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
LOCAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data
OUTPUT_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output

Loading preprocessed data...


AssertionError: macro_df missing expected engineered columns! Found: ['Mkt_Ret', 'Mkt_Ret_Z', 'Macro_Trend', 'Macro_Trend_Z', 'Yield_Curve_10Y2Y_Z', 'Macro_Trend_Vel_Z', 'Macro_Trend_Mom', 'Macro_Vix_Z', 'Macro_Vix_Ratio', 'Mkt_Vol_63d_Z', 'Breadth_Above_SMA50', 'Breadth_Mom_Spread_21d', 'Breadth_CS_Dispersion_21d']

In [ ]:
print("--- macro_df columns ---")
print(macro_df.columns.tolist())
print("\n--- macro_df sample ---")
print(macro_df.tail(2))

print("\n--- features_df columns ---")
print(features_df.columns.tolist())

In [ ]:
# CELL 4: Build Cache
import os
from pathlib import Path
from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR  # Added LOCAL_DATA_DIR

# Create directories if they do not exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Successfully ensured output directory exists: {OUTPUT_DIR}")
print(f"Successfully ensured local data directory exists: {LOCAL_DATA_DIR}")

In [ ]:
# CELL 5: Build Cache
cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()

# Add a warning if the cache file already exists
if cache_file_path.exists():
    print(
        f"Warning: Cache file already exists at {cache_file_path}. Resuming build process."
    )

cache = CheckpointAlphaCache(
    screener=screener, config=config, lookbacks=[CacheConfig.LOOKBACK]
)

cache.build(
    start_date=CacheConfig.START_DATE,
    end_date=None,
    checkpoint_path=cache_file_path,
    save_every_n_days=2000,  # save every ~7 minutes
)

# print(cache.describe())

In [ ]:
print(f"File saved to: {cache_file_path}")
print(f"File exists: {cache_file_path.exists()}")

In [ ]:
# CELL 6: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")

In [ ]:
# CELL 7: Disconnect from TPU
from google.colab import runtime

# This will restart and unassign the current runtime,
# effectively disconnecting from the T4 GPU.
runtime.unassign()
print("Runtime unassigned. This session has been disconnected.")